In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime, timezone

BRONZE_SCHEMA = "supplysage_bronze"
SILVER_SCHEMA = "supplysage_silver"

SOURCE_TABLE = f"{BRONZE_SCHEMA}.bronze_m5_calendar"
TARGET_TABLE = f"{SILVER_SCHEMA}.silver_m5_calendar"

TRANSFORM_RUN_ID = f"silver_m5_calendar_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
CREATED_BY = "Vigya"
NOTEBOOK_NAME = "07_silver_m5_calendar"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

print("TRANSFORM_RUN_ID:", TRANSFORM_RUN_ID)
print("SOURCE_TABLE:", SOURCE_TABLE)
print("TARGET_TABLE:", TARGET_TABLE)

bronze_calendar_df = spark.table(SOURCE_TABLE)

print("Bronze calendar row count:", bronze_calendar_df.count())

display(bronze_calendar_df.limit(10))
bronze_calendar_df.printSchema()

In [0]:
def clean_string_col(col_name):
    return (
        F.when(F.col(col_name).isNull(), None)
        .when(F.trim(F.col(col_name)) == "", None)
        .when(F.lower(F.trim(F.col(col_name))).isin("null", "none", "nan"), None)
        .otherwise(F.trim(F.col(col_name)))
    )


silver_calendar_df = (
    bronze_calendar_df
    .select(
        F.to_date(F.col("date")).alias("calendar_date"),
        F.col("d").alias("m5_day_id"),
        F.col("wm_yr_wk").cast("int").alias("wm_year_week"),
        F.col("weekday").alias("weekday_name"),
        F.col("wday").cast("int").alias("weekday_number"),
        F.col("month").cast("int").alias("month_number"),
        F.col("year").cast("int").alias("calendar_year"),

        clean_string_col("event_name_1").alias("event_name_1"),
        clean_string_col("event_type_1").alias("event_type_1"),
        clean_string_col("event_name_2").alias("event_name_2"),
        clean_string_col("event_type_2").alias("event_type_2"),

        F.col("snap_CA").cast("int").alias("snap_ca"),
        F.col("snap_TX").cast("int").alias("snap_tx"),
        F.col("snap_WI").cast("int").alias("snap_wi")
    )
    .withColumn("calendar_date_id", F.date_format(F.col("calendar_date"), "yyyyMMdd").cast("int"))
    .withColumn("calendar_quarter", F.quarter(F.col("calendar_date")))
    .withColumn("day_of_month", F.dayofmonth(F.col("calendar_date")))
    .withColumn("week_of_year", F.weekofyear(F.col("calendar_date")))
    .withColumn(
        "is_weekend",
        F.when(F.col("weekday_name").isin("Saturday", "Sunday"), F.lit(True)).otherwise(F.lit(False))
    )
    .withColumn(
        "event_count",
        F.when(F.col("event_name_1").isNotNull() & F.col("event_name_2").isNotNull(), F.lit(2))
        .when(F.col("event_name_1").isNotNull() | F.col("event_name_2").isNotNull(), F.lit(1))
        .otherwise(F.lit(0))
    )
    .withColumn("is_event_day", F.col("event_count") > 0)
    .withColumn(
        "is_snap_any_state",
        (F.col("snap_ca") == 1) | (F.col("snap_tx") == 1) | (F.col("snap_wi") == 1)
    )
    .withColumn("silver_transform_run_id", F.lit(TRANSFORM_RUN_ID))
    .withColumn("silver_created_at", F.lit(datetime.now(timezone.utc).isoformat()))
    .withColumn("silver_created_by", F.lit(CREATED_BY))
    .withColumn("silver_notebook_name", F.lit(NOTEBOOK_NAME))
)

print("Silver calendar row count:", silver_calendar_df.count())

display(silver_calendar_df.limit(20))
silver_calendar_df.printSchema()

In [0]:
calendar_validation_rows = []

source_row_count = bronze_calendar_df.count()
target_row_count = silver_calendar_df.count()

calendar_validation_rows.append({
    "validation_check": "row_count_preserved",
    "expected_value": str(source_row_count),
    "actual_value": str(target_row_count),
    "status": "PASS" if source_row_count == target_row_count else "FAIL",
    "issue_detail": None if source_row_count == target_row_count else "Silver calendar row count does not match Bronze calendar row count."
})

duplicate_date_count = (
    silver_calendar_df
    .groupBy("calendar_date")
    .agg(F.count("*").alias("row_count"))
    .filter(F.col("row_count") > 1)
    .count()
)

calendar_validation_rows.append({
    "validation_check": "calendar_date_unique",
    "expected_value": "0 duplicate dates",
    "actual_value": str(duplicate_date_count),
    "status": "PASS" if duplicate_date_count == 0 else "FAIL",
    "issue_detail": None if duplicate_date_count == 0 else "Duplicate calendar_date values found."
})

null_calendar_date_count = (
    silver_calendar_df
    .filter(F.col("calendar_date").isNull())
    .count()
)

calendar_validation_rows.append({
    "validation_check": "calendar_date_not_null",
    "expected_value": "0 null calendar_date",
    "actual_value": str(null_calendar_date_count),
    "status": "PASS" if null_calendar_date_count == 0 else "FAIL",
    "issue_detail": None if null_calendar_date_count == 0 else "Null calendar_date values found."
})

invalid_snap_count = (
    silver_calendar_df
    .filter(
        ~F.col("snap_ca").isin(0, 1)
        | ~F.col("snap_tx").isin(0, 1)
        | ~F.col("snap_wi").isin(0, 1)
    )
    .count()
)

calendar_validation_rows.append({
    "validation_check": "snap_flags_valid",
    "expected_value": "SNAP flags are 0 or 1",
    "actual_value": str(invalid_snap_count),
    "status": "PASS" if invalid_snap_count == 0 else "FAIL",
    "issue_detail": None if invalid_snap_count == 0 else "Invalid SNAP flag values found."
})

invalid_event_count = (
    silver_calendar_df
    .filter(~F.col("event_count").isin(0, 1, 2))
    .count()
)

calendar_validation_rows.append({
    "validation_check": "event_count_valid",
    "expected_value": "event_count is 0, 1, or 2",
    "actual_value": str(invalid_event_count),
    "status": "PASS" if invalid_event_count == 0 else "FAIL",
    "issue_detail": None if invalid_event_count == 0 else "Invalid event_count values found."
})

calendar_validation_schema = T.StructType([
    T.StructField("validation_check", T.StringType(), True),
    T.StructField("expected_value", T.StringType(), True),
    T.StructField("actual_value", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("issue_detail", T.StringType(), True)
])

calendar_validation_df = spark.createDataFrame(
    calendar_validation_rows,
    schema=calendar_validation_schema
)

display(calendar_validation_df.orderBy("status", "validation_check"))

fail_count = calendar_validation_df.filter(F.col("status") == "FAIL").count()

print("Silver calendar validation failures:", fail_count)

In [0]:
if fail_count == 0:
    (
        silver_calendar_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(TARGET_TABLE)
    )

    print(f"✅ Wrote Silver calendar table: {TARGET_TABLE}")
else:
    raise ValueError("Silver calendar validation failed. Fix issues before writing.")

In [0]:
written_calendar_df = spark.table(TARGET_TABLE)

written_row_count = written_calendar_df.count()

print("Written Silver calendar row count:", written_row_count)

display(
    written_calendar_df
    .select(
        "calendar_date_id",
        "calendar_date",
        "m5_day_id",
        "weekday_name",
        "calendar_year",
        "month_number",
        "is_weekend",
        "is_event_day",
        "event_count",
        "is_snap_any_state"
    )
    .orderBy("calendar_date")
    .limit(20)
)

if written_row_count == 1969:
    print("✅ Read-back check passed: Silver calendar table exists and row count is correct.")
else:
    print("❌ Read-back check failed: Row count does not match expected 1969.")

In [0]:
calendar_validation_results_df = (
    calendar_validation_df
    .withColumn("transform_run_id", F.lit(TRANSFORM_RUN_ID))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("target_table", F.lit(TARGET_TABLE))
    .withColumn("validated_at", F.lit(datetime.now(timezone.utc).isoformat()))
    .withColumn("created_by", F.lit(CREATED_BY))
    .withColumn("notebook_name", F.lit(NOTEBOOK_NAME))
    .select(
        "transform_run_id",
        "source_table",
        "target_table",
        "validation_check",
        "expected_value",
        "actual_value",
        "status",
        "issue_detail",
        "validated_at",
        "created_by",
        "notebook_name"
    )
)

validation_results_table = f"{SILVER_SCHEMA}.silver_transform_validation_results"

try:
    spark.sql(f"""
    DELETE FROM {validation_results_table}
    WHERE transform_run_id = '{TRANSFORM_RUN_ID}'
    """)
except Exception as e:
    print("Validation results table does not exist yet. It will be created now.")

(
    calendar_validation_results_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(validation_results_table)
)

display(
    spark.table(validation_results_table)
    .filter(F.col("transform_run_id") == TRANSFORM_RUN_ID)
    .groupBy("status")
    .agg(F.count("*").alias("validation_check_count"))
    .orderBy("status")
)

print("✅ Notebook 07 PASSED: silver_m5_calendar created and validated.")
print("Next notebook: 08_silver_m5_sell_prices")